# LeetCode 2138 — Divide a String Into Groups of Size k bằng Python

Link đề: https://leetcode.com/problems/divide-a-string-into-groups-of-size-k/ — Độ khó: **Easy**

Skeleton LeetCode:

```python
class Solution:
    def divideString(self, s: str, k: int, fill: str) -> List[str]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu cách **chia chuỗi thành các khối** độ dài `k` và **đệm** (`fill`) khối cuối nếu thiếu.
2. Thành thạo slicing `s[i:i+k]`, `range(0, n, k)` (bước nhảy), `ljust` và nhân chuỗi `fill * m`.
3. Viết 4 cách: duyệt từng ký tự gom nhóm, cắt lát theo bước `k`, đệm trước rồi cắt, và dùng phép chia lấy dư.
4. Hiểu vì sao slicing Python **không báo lỗi** khi vượt biên và cách tận dụng điều đó.
5. Tránh các lỗi hay gặp: quên đệm, đệm thừa khi chia hết, vòng lặp sai bước.

## 2. Đề bài & ví dụ

Cho chuỗi `s`, số nguyên `k` và ký tự `fill`. Chia `s` thành các nhóm liên tiếp, mỗi nhóm đúng `k` ký tự, theo thứ tự từ trái sang phải. Nếu nhóm **cuối** thiếu ký tự thì đệm thêm `fill` vào **cuối nhóm** cho đủ `k` ký tự. Trả về mảng các nhóm.

| s | k | fill | kết quả |
|---|---|---|---|
| `"abcdefghi"` | 3 | `"x"` | `["abc","def","ghi"]` |
| `"abcdefghij"` | 3 | `"x"` | `["abc","def","ghi","jxx"]` |

Constraints: `1 <= s.length <= 100`, `s` chỉ gồm chữ thường, `1 <= k <= 100`, `fill` là một chữ thường.

Ghi nhớ: nếu `len(s)` chia hết cho `k` thì **không** thêm nhóm nào và không đệm gì cả.

In [1]:
examples = [
    ("abcdefghi", 3, "x", ["abc", "def", "ghi"]),
    ("abcdefghij", 3, "x", ["abc", "def", "ghi", "jxx"]),
]
for s, k, fill, expected in examples:
    print(f"s={s!r}, k={k}, fill={fill!r}, len={len(s)}, len%k={len(s) % k} -> {expected}")

s='abcdefghi', k=3, fill='x', len=9, len%k=0 -> ['abc', 'def', 'ghi']
s='abcdefghij', k=3, fill='x', len=10, len%k=1 -> ['abc', 'def', 'ghi', 'jxx']


## 3. Kiến thức nền

### 3.1. Slicing `s[a:b]` và độ dài

`s[a:b]` lấy các ký tự có chỉ số từ `a` đến `b-1` (**không** gồm `b`). Độ dài nhóm là `b - a`. Vì thế `s[i:i+k]` luôn là nhóm `k` ký tự bắt đầu từ `i` (nếu đủ chỗ).

In [2]:
s = "abcdefghij"
print(s[0:3], s[3:6], s[6:9], s[9:12])

abc def ghi j


Dòng cuối cho thấy `s[9:12]` chỉ trả `"j"` mà **không lỗi** — slicing tự động cắt theo biên chuỗi. Đây là điểm mấu chốt của bài: nhóm cuối tự nhiên **ngắn hơn** `k`, và ta chỉ cần đệm thêm.

### 3.2. `range(start, stop, step)` với bước `k`

`range(0, n, k)` sinh `0, k, 2k, ...` nhỏ hơn `n`: đúng các chỉ số bắt đầu của từng nhóm.

In [3]:
print(list(range(0, 10, 3)))
print(list(range(0, 9, 3)))
print(list(range(0, 3, 5)))

[0, 3, 6, 9]
[0, 3, 6]
[0]


### 3.3. Nhân chuỗi `fill * m` và `ljust`

- `"x" * 2` → `"xx"`; `"x" * 0` → `""` (chuỗi rỗng).
- `"j".ljust(3, "x")` → `"jxx"`: căn trái, đệm bằng `"x"` cho đủ chiều rộng 3; nếu đã đủ hoặc dài hơn thì giữ nguyên.

In [4]:
print("x" * 2, repr("x" * 0))
print("j".ljust(3, "x"), "abc".ljust(3, "x"), "abcd".ljust(3, "x"))

xx ''
jxx abc abcd


### 3.4. Phép chia lấy dư `%`

`len(s) % k` là số ký tự **dư** ở nhóm cuối khi chia hết thì bằng 0. Số ký tự cần đệm là `(k - len(s) % k) % k`: phép `% k` ngoài cùng biến trường hợp "đủ rồi" (`k - 0 = k`) thành `0`.

In [5]:
k = 3
for n in range(7, 13):
    print(f"n={n:2d}, n%k={n % k}, cần đệm={(k - n % k) % k}")

n= 7, n%k=1, cần đệm=2
n= 8, n%k=2, cần đệm=1
n= 9, n%k=0, cần đệm=0
n=10, n%k=1, cần đệm=2
n=11, n%k=2, cần đệm=1
n=12, n%k=0, cần đệm=0


### 3.5. List comprehension với `range`

`[s[i:i+k] for i in range(0, n, k)]` gom mọi nhóm trong một dòng.

In [6]:
s = "abcdefghij"
print([s[i:i + 3] for i in range(0, len(s), 3)])

['abc', 'def', 'ghi', 'j']


## 4. Ý tưởng nền tảng — tính bằng tay

`s = "abcdefghij"` (10 ký tự), `k = 3`, `fill = "x"`:

- Nhóm 1: ký tự 0–2: `abc`.
- Nhóm 2: ký tự 3–5: `def`.
- Nhóm 3: ký tự 6–8: `ghi`.
- Nhóm 4: chỉ còn ký tự 9 là `j`. Thiếu `3 - 1 = 2` ký tự → đệm `xx` → `jxx`.

Số nhóm = `ceil(10 / 3) = 4`. Số ký tự đệm = `4*3 - 10 = 2`.

In [7]:
s, k, fill = "abcdefghij", 3, "x"
n = len(s)
groups = -(-n // k)          # ceil(n / k) bằng phép chia làm tròn lên
pad = groups * k - n
print("số nhóm =", groups, "| số ký tự đệm =", pad)

số nhóm = 4 | số ký tự đệm = 2


## 5. Cách 1 — Brute Force: duyệt từng ký tự và gom nhóm

Cầm một nhóm tạm `cur`. Mỗi ký tự thêm vào `cur`; khi `cur` đủ `k` ký tự thì đẩy vào kết quả và làm rỗng. Cuối cùng nếu `cur` còn dở thì đệm và đẩy nốt.

In [8]:
def divide_char_by_char(s, k, fill):
    groups = []
    cur = ""
    for ch in s:
        cur += ch
        if len(cur) == k:
            groups.append(cur)
            cur = ""
    if cur:                       # nhóm cuối còn dở
        while len(cur) < k:
            cur += fill
        groups.append(cur)
    return groups

for s, k, fill, expected in examples:
    print(divide_char_by_char(s, k, fill), expected)

['abc', 'def', 'ghi'] ['abc', 'def', 'ghi']
['abc', 'def', 'ghi', 'jxx'] ['abc', 'def', 'ghi', 'jxx']


Độ phức tạp: mỗi ký tự được xử lý `O(1)` (nhóm dài ≤ 100) → **Time `O(n)`** (với `n = len(s)`; nối chuỗi từng ký tự nhỏ nên coi như `O(n·k)` tệ nhất `O(n)` vì `k ≤ 100`), **Space `O(n)`** cho kết quả. Nút thắt: code dài, phải xử lý riêng nhóm cuối, dễ sai.

## 6. Ý tưởng thuật toán / bước đột phá

Hai nhận xét:

1. Nhóm thứ `j` (đánh số từ 0) chính là `s[j*k : j*k + k]` — nên chỉ cần lặp `i = 0, k, 2k, ...`.
2. Slicing vượt biên tự cắt, nên **nhóm cuối đã đúng phần ký tự có thật**; chỉ cần đệm thêm `fill` cho đủ `k`.

Quy tắc "nếu… thì…": nếu `len(s) % k == 0` thì không cần đệm; ngược lại chỉ nhóm cuối cần đệm thêm `k - len(s) % k` ký tự. `ljust(k, fill)` làm đúng việc này cho **mọi** nhóm (nhóm đủ thì không đổi), nên không cần `if`.

## 7. Các cách tiếp theo

### Cách 2 — Cắt lát theo bước `k` + `ljust`

Với mỗi `i` trong `range(0, n, k)`, nhóm là `s[i:i+k].ljust(k, fill)`.

In [9]:
def divide_slice_ljust(s, k, fill):
    return [s[i:i + k].ljust(k, fill) for i in range(0, len(s), k)]

for s, k, fill, expected in examples:
    print(divide_slice_ljust(s, k, fill), expected)

['abc', 'def', 'ghi'] ['abc', 'def', 'ghi']
['abc', 'def', 'ghi', 'jxx'] ['abc', 'def', 'ghi', 'jxx']


In [10]:
from typing import List

class SolutionSlice:
    def divideString(self, s: str, k: int, fill: str) -> List[str]:
        return [s[i:i + k].ljust(k, fill) for i in range(0, len(s), k)]

print(SolutionSlice().divideString("abcdefghij", 3, "x"))

['abc', 'def', 'ghi', 'jxx']


Độ phức tạp: **Time `O(n)`**, **Space `O(n)`**.

### Cách 3 — Đệm `s` trước, rồi cắt đều

Tính số ký tự cần đệm `pad = (k - len(s) % k) % k`, tạo `s2 = s + fill * pad` (độ dài giờ chia hết cho `k`), rồi cắt theo bước `k` — không cần `ljust`, không có nhóm ngắn.

In [11]:
def divide_pad_first(s, k, fill):
    pad = (k - len(s) % k) % k
    s2 = s + fill * pad
    return [s2[i:i + k] for i in range(0, len(s2), k)]

for s, k, fill, expected in examples:
    print(divide_pad_first(s, k, fill), expected)

['abc', 'def', 'ghi'] ['abc', 'def', 'ghi']
['abc', 'def', 'ghi', 'jxx'] ['abc', 'def', 'ghi', 'jxx']


In [12]:
from typing import List

class SolutionPadFirst:
    def divideString(self, s: str, k: int, fill: str) -> List[str]:
        s += fill * ((k - len(s) % k) % k)
        return [s[i:i + k] for i in range(0, len(s), k)]

print(SolutionPadFirst().divideString("abcdefghi", 3, "x"))

['abc', 'def', 'ghi']


Độ phức tạp: **Time `O(n)`**, **Space `O(n)`**. Ưu điểm: logic đệm tách bạch, dễ kiểm tra bằng assert `len(s2) % k == 0`.

### Cách 4 — Tính số nhóm rồi dựng từng nhóm bằng chỉ số

Số nhóm `g = (n + k - 1) // k` (chia làm tròn lên bằng số nguyên). Với nhóm `j`, ký tự thứ `t` (0 ≤ t < k) là `s[j*k + t]` nếu `j*k + t < n`, ngược lại là `fill`. Cách này giống một vòng lặp lồng nhau ở C/Java — hữu ích khi không có slicing.

In [13]:
def divide_index_math(s, k, fill):
    n = len(s)
    g = (n + k - 1) // k
    groups = []
    for j in range(g):
        chars = []
        for t in range(k):
            p = j * k + t
            chars.append(s[p] if p < n else fill)
        groups.append("".join(chars))
    return groups

for s, k, fill, expected in examples:
    print(divide_index_math(s, k, fill), expected)

['abc', 'def', 'ghi'] ['abc', 'def', 'ghi']
['abc', 'def', 'ghi', 'jxx'] ['abc', 'def', 'ghi', 'jxx']


In [14]:
from typing import List

class SolutionIndex:
    def divideString(self, s: str, k: int, fill: str) -> List[str]:
        n = len(s)
        res = []
        for j in range((n + k - 1) // k):
            res.append("".join(s[p] if p < n else fill for p in range(j * k, j * k + k)))
        return res

print(SolutionIndex().divideString("abcdefghij", 3, "x"))

['abc', 'def', 'ghi', 'jxx']


Độ phức tạp: **Time `O(n + k)`** (số nhóm × `k`, tối đa `n + k - 1` ký tự được dựng), **Space `O(n)`**.

## 8. TRACE từng bước

Trace Cách 2 với `s = "abcdefghij"`, `k = 3`, `fill = "x"`:

| i | `s[i:i+k]` | độ dài | sau `ljust` | nhóm |
|---|---|---|---|---|
| 0 | `abc` | 3 | `abc` | `abc` |
| 3 | `def` | 3 | `def` | `def` |
| 6 | `ghi` | 3 | `ghi` | `ghi` |
| 9 | `j` | 1 | đệm 2 → `jxx` | `jxx` |

In [15]:
def trace_slice(s, k, fill):
    groups = []
    for i in range(0, len(s), k):
        piece = s[i:i + k]
        padded = piece.ljust(k, fill)
        groups.append(padded)
        print(f"i={i}: s[{i}:{i + k}]={piece!r} (dài {len(piece)}) -> sau ljust {padded!r}")
    print("kết quả:", groups)
    return groups

trace_slice("abcdefghij", 3, "x")

i=0: s[0:3]='abc' (dài 3) -> sau ljust 'abc'
i=3: s[3:6]='def' (dài 3) -> sau ljust 'def'
i=6: s[6:9]='ghi' (dài 3) -> sau ljust 'ghi'
i=9: s[9:12]='j' (dài 1) -> sau ljust 'jxx'
kết quả: ['abc', 'def', 'ghi', 'jxx']


['abc', 'def', 'ghi', 'jxx']

In [16]:
def trace_char(s, k, fill):
    groups, cur = [], ""
    for i, ch in enumerate(s):
        cur += ch
        print(f"i={i}, ch={ch!r}, cur={cur!r}")
        if len(cur) == k:
            groups.append(cur)
            print("   -> đủ k, đẩy nhóm", groups)
            cur = ""
    if cur:
        print("còn dở:", repr(cur), "-> đệm", repr(fill * (k - len(cur))))
        groups.append(cur + fill * (k - len(cur)))
    return groups

print(trace_char("abcde", 2, "z"))

i=0, ch='a', cur='a'
i=1, ch='b', cur='ab'
   -> đủ k, đẩy nhóm ['ab']
i=2, ch='c', cur='c'
i=3, ch='d', cur='cd'
   -> đủ k, đẩy nhóm ['ab', 'cd']
i=4, ch='e', cur='e'
còn dở: 'e' -> đệm 'z'
['ab', 'cd', 'ez']


## 9. Code nộp LeetCode

In [17]:
from typing import List

class Solution:
    def divideString(self, s: str, k: int, fill: str) -> List[str]:
        return [s[i:i + k].ljust(k, fill) for i in range(0, len(s), k)]

print(Solution().divideString("abcdefghi", 3, "x"))
print(Solution().divideString("abcdefghij", 3, "x"))

['abc', 'def', 'ghi']
['abc', 'def', 'ghi', 'jxx']


## 10. Vì sao thuật toán đúng?

**Các nhóm phủ đúng `s`, không chồng lấn.** Các chỉ số bắt đầu `0, k, 2k, ...` cách nhau đúng `k`, nên các khoảng `[ik, ik+k)` rời nhau và nối tiếp, phủ hết `[0, n)` (nhóm cuối bị cắt ở `n`). Do đó ghép mọi nhóm (trước khi đệm) lại cho đúng `s`.

**Đệm đúng.** Mọi nhóm trừ nhóm cuối có đúng `k` ký tự (vì còn đủ chỗ trong `s`) nên `ljust` không làm gì. Nhóm cuối có `r = n - (g-1)k` ký tự, `1 ≤ r ≤ k`; `ljust(k, fill)` thêm `k - r` ký tự `fill` vào **cuối** — đúng yêu cầu. Khi `n` chia hết cho `k`, `r = k` và không có đệm.

**Số nhóm đúng.** `range(0, n, k)` có đúng `ceil(n/k)` phần tử, tức không thiếu nhóm và không thừa nhóm rỗng.

### Kiểm chứng ba khẳng định của phần chứng minh bằng code

Ta không chỉ tin lập luận bằng lời; hãy để máy kiểm tra ba điều: (a) số nhóm bằng `ceil(n/k)`, (b) chỉ nhóm cuối có thể bị đệm, (c) số ký tự đệm bằng `(k - n % k) % k`. Với mọi `n` từ 1 đến 60 và mọi `k` từ 1 đến 20 ta đối chiếu trực tiếp.

In [18]:
bad = 0
for n in range(1, 61):
    for k in range(1, 21):
        s = "a" * n
        groups = Solution().divideString(s, k, "z")
        assert len(groups) == -(-n // k)                          # (a)
        assert all("z" not in g for g in groups[:-1])             # (b)
        padded = groups[-1].count("z")
        assert padded == (k - n % k) % k                          # (c)
        bad += 0
print("ba khẳng định đúng với mọi 1 <= n <= 60 và 1 <= k <= 20")

ba khẳng định đúng với mọi 1 <= n <= 60 và 1 <= k <= 20


## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt từng ký tự | `O(n)` | `O(n)` |
| 2. Slice + `ljust` | `O(n)` | `O(n)` |
| 3. Đệm trước rồi cắt | `O(n)` | `O(n)` |
| 4. Chỉ số toán học | `O(n + k)` | `O(n)` |

Mọi cách phải đọc `n` ký tự và xuất ra ít nhất `n` ký tự, nên `Ω(n)` là cận dưới; cả bốn cách đạt cận đó. Không có khác biệt worst/average. Space `O(n)` là bộ nhớ của kết quả — không thể tránh.

## 12. Bộ test

Kiểm ví dụ đề, edge case (`k = 1`, `k >= n`, `n` chia hết `k`, `n = 1`), kiểm tính chất tổng quát (độ dài mọi nhóm là `k`, ghép lại bắt đầu bằng `s`, phần đệm toàn `fill`) và đối chiếu mọi cách bằng input ngẫu nhiên có seed.

In [19]:
cases = [
    ("abcdefghi", 3, "x", ["abc", "def", "ghi"]),
    ("abcdefghij", 3, "x", ["abc", "def", "ghi", "jxx"]),
    ("a", 1, "z", ["a"]),
    ("a", 5, "z", ["azzzz"]),
    ("abc", 5, "q", ["abcqq"]),
    ("abcd", 2, "y", ["ab", "cd"]),
    ("abcde", 2, "y", ["ab", "cd", "ey"]),
    ("abcdef", 1, "w", ["a", "b", "c", "d", "e", "f"]),
    ("abcdef", 6, "w", ["abcdef"]),
    ("abcdef", 7, "w", ["abcdefw"]),
]
impls = [divide_char_by_char, divide_slice_ljust, divide_pad_first, divide_index_math,
         Solution().divideString, SolutionSlice().divideString,
         SolutionPadFirst().divideString, SolutionIndex().divideString]
for s, k, fill, expected in cases:
    for f in impls:
        assert f(s, k, fill) == expected, (f.__name__, s, k, fill, f(s, k, fill))
print("Tất cả", len(cases), "ca x", len(impls), "cách đều đúng")

Tất cả 10 ca x 8 cách đều đúng


In [20]:
import random
random.seed(2138)
for _ in range(3000):
    n = random.randint(1, 40)
    s = "".join(random.choice("abcdefgh") for _ in range(n))
    k = random.randint(1, 12)
    fill = random.choice("xyz")
    want = divide_char_by_char(s, k, fill)
    for f in impls[1:]:
        assert f(s, k, fill) == want, (f.__name__, s, k, fill)
    # tính chất
    assert all(len(g) == k for g in want)
    joined = "".join(want)
    assert joined.startswith(s) and set(joined[n:]) <= {fill}
    assert len(want) == -(-n // k)
print("3000 input ngẫu nhiên: các cách khớp, mọi nhóm dài k, phần đệm chỉ là fill")

3000 input ngẫu nhiên: các cách khớp, mọi nhóm dài k, phần đệm chỉ là fill


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên đệm nhóm cuối

Chỉ cắt lát, nhóm cuối `"j"` ngắn hơn `k`.

In [21]:
def wrong1(s, k, fill):
    return [s[i:i + k] for i in range(0, len(s), k)]

print(wrong1("abcdefghij", 3, "x"), "| đúng:", Solution().divideString("abcdefghij", 3, "x"))

['abc', 'def', 'ghi', 'j'] | đúng: ['abc', 'def', 'ghi', 'jxx']


### Lỗi 2: Đệm **thừa** khi `n` chia hết cho `k`

Dùng `pad = k - len(s) % k` (không có `% k` ngoài). Khi `n % k == 0`, `pad = k` → thêm cả một nhóm toàn `fill` không mong muốn.

In [22]:
def wrong2(s, k, fill):
    s2 = s + fill * (k - len(s) % k)       # SAI khi len(s) % k == 0
    return [s2[i:i + k] for i in range(0, len(s2), k)]

print(wrong2("abcdefghi", 3, "x"), "| đúng:", Solution().divideString("abcdefghi", 3, "x"))

['abc', 'def', 'ghi', 'xxx'] | đúng: ['abc', 'def', 'ghi']


### Lỗi 3: `range(0, len(s), 1)` hoặc quên bước `k`

Dùng `range(len(s))` rồi cắt `s[i:i+k]` sẽ cho các nhóm **chồng lấn** và quá nhiều nhóm.

In [23]:
def wrong3(s, k, fill):
    return [s[i:i + k].ljust(k, fill) for i in range(len(s))]       # thiếu bước k

print(wrong3("abcde", 2, "y"))
print(Solution().divideString("abcde", 2, "y"))

['ab', 'bc', 'cd', 'de', 'ey']
['ab', 'cd', 'ey']


### Lỗi 4: Đệm vào đầu nhóm thay vì cuối

`rjust` đệm bên trái; đề yêu cầu đệm **cuối** nên phải dùng `ljust`.

In [24]:
def wrong4(s, k, fill):
    return [s[i:i + k].rjust(k, fill) for i in range(0, len(s), k)]

print(wrong4("abcdefghij", 3, "x"), "| đúng:", Solution().divideString("abcdefghij", 3, "x"))

['abc', 'def', 'ghi', 'xxj'] | đúng: ['abc', 'def', 'ghi', 'jxx']


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chia thành các khối kích thước `k`" → `range(0, n, k)` + `s[i:i+k]`. Hãy học thuộc mẫu này, nó xuất hiện rất nhiều (chunking, batch, phân trang).
- Slicing vượt biên **không lỗi**; tận dụng để không phải xử lý riêng nhóm cuối.
- `ljust`/`rjust`/`center` giúp đệm chuỗi trong một lệnh; `fill * m` cũng đệm được.
- Số khối = `ceil(n / k) = (n + k - 1) // k`; số ký tự đệm = `(k - n % k) % k`.
- Tự thêm assert: mọi khối có độ dài `k`, tổng ký tự `= số khối * k`.

### Một ví dụ thực tế của "chunking"

Mẫu `range(0, n, k)` + `s[i:i+k]` là xương sống của nhiều tác vụ đời thực: chia danh sách người dùng thành các lô (batch) 100 người để gửi email, chia file lớn thành các khối 4 KB, hay in số điện thoại thành nhóm 3 chữ số (`0123456789` → `012 345 678 9`). Ví dụ dưới đây gom nhóm một chuỗi số và nối bằng dấu cách để dễ đọc.

In [25]:
def pretty(digits, k=3):
    return " ".join(digits[i:i + k] for i in range(0, len(digits), k))

print(pretty("0123456789"))
print(pretty("1234567", 2))

012 345 678 9
12 34 56 7


## 15. Biến thể của bài

1. Đệm vào **đầu** nhóm đầu tiên thay vì cuối nhóm cuối.
2. Chia thành **đúng `m` nhóm** gần bằng nhau thay vì mỗi nhóm `k` ký tự.
3. Bỏ nhóm cuối nếu thiếu (không đệm).
4. Chia **mảng** (list) thành các khối kích thước `k`, đệm bằng một giá trị mặc định.
5. Trả về **chỉ số bắt đầu** của từng nhóm.

In [26]:
def divide_drop_last(s, k):
    return [s[i:i + k] for i in range(0, len(s) - len(s) % k, k)]

def chunk_list(a, k, pad):
    return [(a[i:i + k] + [pad] * k)[:k] for i in range(0, len(a), k)]

def group_starts(n, k):
    return list(range(0, n, k))

print(divide_drop_last("abcdefghij", 3))
print(chunk_list([1, 2, 3, 4, 5], 2, 0))
print(group_starts(10, 3))

['abc', 'def', 'ghi']
[[1, 2], [3, 4], [5, 0]]
[0, 3, 6, 9]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt từng ký tự | gom `cur` đủ `k` thì đẩy | `O(n)` | `O(n)` | học, không dùng slicing |
| 2. Slice + `ljust` | `s[i:i+k].ljust(k, fill)` | `O(n)` | `O(n)` | nộp bài, ngắn nhất |
| 3. Đệm trước rồi cắt | `s += fill * pad` | `O(n)` | `O(n)` | dễ kiểm chứng độ dài |
| 4. Chỉ số toán học | `s[j*k+t]` hoặc `fill` | `O(n+k)` | `O(n)` | ngôn ngữ không có slicing |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `divide_slice_ljust("hello", 2, "z")` in ra gì?

**Bài 2 (tính tay):** `s = "leetcode"` (8 ký tự), `k = 5`, `fill = "!"`: có mấy nhóm, cần đệm mấy ký tự, kết quả?

**Bài 3 (điền TODO):** hoàn thành `exercise_pad_count(n, k)` trả về số ký tự đệm.

**Bài 4 (tự viết lại):** viết `exercise_divide(s, k, fill)` bằng "đệm trước rồi cắt".

**Bài 5 (mini challenge):** viết `exercise_left_pad(s, k, fill)` đệm vào **đầu** chuỗi để `len` chia hết cho `k`, rồi cắt thành các nhóm.

In [27]:
# Bài 1: chạy để so với dự đoán
print(divide_slice_ljust("hello", 2, "z"))

['he', 'll', 'oz']


In [28]:
def exercise_pad_count(n, k):
    # TODO: (k - n % k) % k
    pass

def exercise_divide(s, k, fill):
    # TODO: tính pad, s2 = s + fill * pad, rồi cắt theo bước k
    pass

def exercise_left_pad(s, k, fill):
    # TODO: tương tự nhưng đệm vào ĐẦU: s2 = fill * pad + s
    pass

# (không gọi khi chưa làm xong)

## 18. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | `hello` có 5 ký tự, chia 2 | Các nhóm: `he`, `ll`, `o` | `['he', 'll', 'oz']` |
| 2 | `8 / 5` làm tròn lên | 2 nhóm → tổng 10 ký tự | `['leetc', 'ode!!']`, đệm 2 |
| 3 | Khi `n % k == 0` không đệm | Tránh `k - n % k` thuần | `(k - n % k) % k` |
| 4 | Đệm xong thì `len(s2) % k == 0` | `range(0, len(s2), k)` | `[s2[i:i+k] for i in ...]` |
| 5 | Chỉ đổi chỗ phần đệm | `fill * pad + s` | Cắt theo bước `k` như cũ |

## 19. Đáp án tham khảo cho bài tập

Bài 1: `['he', 'll', 'oz']`. Bài 2: 2 nhóm, đệm 2 ký tự: `['leetc', 'ode!!']`.

In [29]:
def exercise_pad_count(n, k):
    return (k - n % k) % k

def exercise_divide(s, k, fill):
    s2 = s + fill * exercise_pad_count(len(s), k)
    return [s2[i:i + k] for i in range(0, len(s2), k)]

def exercise_left_pad(s, k, fill):
    s2 = fill * exercise_pad_count(len(s), k) + s
    return [s2[i:i + k] for i in range(0, len(s2), k)]

assert divide_slice_ljust("hello", 2, "z") == ["he", "ll", "oz"]
assert exercise_divide("leetcode", 5, "!") == ["leetc", "ode!!"]
assert exercise_pad_count(8, 5) == 2 and exercise_pad_count(9, 3) == 0
assert exercise_pad_count(1, 4) == 3
assert exercise_divide("abcdefghi", 3, "x") == ["abc", "def", "ghi"]
assert exercise_left_pad("abcde", 3, "-") == ["-ab", "cde"]
assert exercise_left_pad("abc", 3, "-") == ["abc"]
print("Đáp án bài tập đều đúng")

Đáp án bài tập đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (chunking):**

```python
return [s[i:i + k].ljust(k, fill) for i in range(0, len(s), k)]
```

**Checklist:**

- [ ] Bước nhảy của `range` là `k` (không phải 1)?
- [ ] Nhóm cuối được đệm bằng `fill` ở **cuối** (`ljust`)?
- [ ] Khi `len(s) % k == 0` không sinh thêm nhóm/đệm thừa?
- [ ] Đã thử `k = 1`, `k > len(s)`, `len(s) == 1`?
- [ ] Mọi nhóm trong kết quả đều có đúng `k` ký tự?